Before starting, follow this four steps to run the notebook on Google Colab

Step 1. Clone GitHub repository with notebooks and data for the course

In [10]:
!git clone -- https://github.com/paulmunozpauta/Hidrology_Course

Cloning into 'Hidrology_Course'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 18 (delta 2), reused 14 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 18.08 MiB | 21.56 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [11]:
ls

Hidrology_Course/  sample_data/


Step 2: Let's move to the cloned folder


In [12]:
%cd Hydrology_course

[Errno 2] No such file or directory: 'Hydrology_course'
/content


In [ ]:
ls

# Delimitación de cuenca hidrográfica

Este notebook permite delimitar una cuenca hidrográfica a partir de:

- un DEM (`dem.tif`)
- un punto de salida definido con coordenadas `lon/lat`

Está pensado para ejecutarse en **Google Colab** con librerías comunes y código simple, claro y bien comentado.


In [1]:
# Instalar librerías necesarias
!pip -q install leafmap pysheds rasterio geopandas shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 666.8/666.8 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.9 MB/s eta 0:00:00


In [2]:
# Importar librerías
import os
import numpy as np
import matplotlib.pyplot as plt

import rasterio
from rasterio.features import shapes
from shapely.geometry import shape
import geopandas as gpd

from pysheds.grid import Grid
import leafmap.foliumap as leafmap

## 1. Mostrar mapa base

Usa este mapa solo para ubicar visualmente la zona de estudio y elegir el punto de salida.
En esta primera versión, las coordenadas del punto se ingresan manualmente.


In [4]:
# Crear mapa base
m = leafmap.Map()
m.add_basemap("OpenTopoMap")
m

In [5]:
# Crear mapa base en un punto específico
m = leafmap.Map(center=[-36.59433389487894, -72.08269046506874], zoom=15)
m.add_basemap("OpenTopoMap")
m

## 2. Definir la ruta del DEM

Antes de ejecutar esta parte, sube a Colab un archivo llamado `dem.tif`.


DEM oficial Hidrografía Chile (DGA)
https://lineasdebasepublicas.mma.gob.cl/datos_abiertos/dataset/hidrosfera/resource/0c159d9e-4378-40f0-80da-240aad755a21?utm_source=chatgpt.com

In [8]:
# Ruta del DEM
dem_path = "/content/Data/Ñuble/REGION_NUBLE_1.jp2"

if not os.path.exists(dem_path):
    raise FileNotFoundError(
        "No se encontró el archivo DEM"
    )

print("DEM encontrado:", dem_path)

FileNotFoundError: No se encontró el archivo DEM en /content/dem.tif. 

## 3. Definir el punto de salida

Reemplaza estos valores por las coordenadas del punto ubicado sobre el río.


In [ ]:
# Coordenadas del punto de salida
outlet_lon = -72.95
outlet_lat = -36.82

print(f"Punto de salida: lon={outlet_lon}, lat={outlet_lat}")

## 4. Cargar y preparar el DEM

Aquí se corrigen depresiones y zonas planas para que el flujo superficial pueda calcularse correctamente.


In [ ]:
# Crear la grilla a partir del DEM
grid = Grid.from_raster(dem_path)
dem = grid.read_raster(dem_path)

print("Rellenando depresiones...")
dem_filled = grid.fill_depressions(dem)

print("Resolviendo zonas planas...")
dem_inflated = grid.resolve_flats(dem_filled)

print("DEM listo para el análisis hidrológico.")

## 5. Calcular dirección y acumulación de flujo


In [ ]:
# Calcular dirección de flujo
fdir = grid.flowdir(dem_inflated)

# Calcular acumulación de flujo
acc = grid.accumulation(fdir)

print("Dirección y acumulación de flujo calculadas.")

## 6. Convertir coordenadas geográficas a fila y columna del raster


In [ ]:
with rasterio.open(dem_path) as src:
    row, col = src.index(outlet_lon, outlet_lat)

print(f"Fila: {row}, Columna: {col}")

## 7. Delimitar la cuenca


In [ ]:
# Delimitar cuenca usando el punto de salida
catch = grid.catchment(
    x=col,
    y=row,
    fdir=fdir,
    xytype="index"
)

# Recortar la grilla a la cuenca
grid.clip_to(catch)
catch_view = grid.view(catch)

print("Cuenca delimitada correctamente.")

## 8. Visualizar la cuenca resultante


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(catch_view, extent=grid.extent, cmap="Blues")
ax.set_title("Cuenca delimitada")
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
plt.show()

## 9. Convertir la cuenca a polígono

Esto permite exportar el resultado a formatos GIS como GeoJSON o Shapefile.


In [ ]:
# Convertir la máscara raster a polígonos
mask = catch_view.astype(np.uint8)

results = shapes(mask, mask=mask > 0, transform=grid.affine)
geoms = [shape(geom) for geom, value in results if value == 1]

if len(geoms) == 0:
    raise ValueError("No se pudo generar el polígono de la cuenca.")

# Crear GeoDataFrame y disolver geometrías
gdf = gpd.GeoDataFrame(geometry=geoms, crs="EPSG:4326")
gdf = gdf.dissolve()

gdf

## 10. Guardar resultados


In [ ]:
output_geojson = "/content/cuenca_delimitada.geojson"
output_shp = "/content/cuenca_delimitada.shp"

gdf.to_file(output_geojson, driver="GeoJSON")
gdf.to_file(output_shp)

print("Archivos guardados:")
print(output_geojson)
print(output_shp)

## Nota

Esta es una primera versión simple. En la siguiente mejora se puede agregar:

- selección del punto con clic en el mapa
- ajuste automático del punto al cauce más cercano
- visualización del polígono directamente sobre el mapa
- carga automática de DEM desde una fuente abierta
